In [57]:
import cv2
import pytesseract
import numpy as np
import os
import re
from datetime import datetime
pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'
def proc_img(img_path):
    img = cv2.imread(img_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, thres_img = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY)
    return thres_img, img
def get_txt(img_path):
    thres_img, img = proc_img(img_path)
    nums = find_nums(img)
    txt = pytesseract.image_to_string(thres_img)
    return txt, nums
def find_nums(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    circ = cv2.HoughCircles(blurred, cv2.HOUGH_GRADIENT, dp=1.2, minDist=50, param1=50, param2=30, minRadius=10, maxRadius=50)
    nums = []
    if circ is not None:
        circ = np.round(circ[0, :]).astype("int")
        for (x, y, r) in circ:
            y1, y2 = max(0, y - r), min(img.shape[0], y + r)
            x1, x2 = max(0, x - r), min(img.shape[1], x + r)
            roi = img[y1:y2, x1:x2]
            txt = pytesseract.image_to_string(roi, config='--psm 6')
            found_nums = re.findall(r'\d+', txt)
            if found_nums:
                nums.append(int(found_nums[0]))
    return nums
def fmt_date(raw_date):
    try:
        date_obj = datetime.strptime(raw_date, "%d/%m/%Y")
        return date_obj.strftime("%d/%m/%Y")
    except ValueError:
        return None
def parse(txt, nums):
    data = {
        "patient_name": "",
        "dob": "",
        "date": "",
        "injection": "",
        "exercise_therapy": "",
        "difficulty_ratings": {},
        "patient_changes": {},
        "pain_symptoms": {},
        "medical_assistant_data": {},
        "filled_by_ma": ""
    }
    difficulty_keys = ["bending", "putting on shoes", "sleeping", "standing", "walking", "driving", "preparing", "yard work", "picking up items"]
    pain_keys = ["pain", "numbness", "tingling", "burning", "tightness"]
    
    data["medical_assistant_data filled ny Ma"] = {
        "blood_pressure": "135/89",
        "hr": 90,
        "weight": 90,
        "height": "5'8",
        "spo2": 88,
        "temperature": "97.6",
        "blood_glucose": 163,
        "respirations": 15
    }
    lines = txt.split('\n')
    for line in lines:
        line = line.strip()
        if "Patient Name" in line:
            data['patient_name'] = line.split(':')[-1].strip()
        elif "DOB" in line:
            match = re.search(r'(\d{2}/\d{2}/\d{4})', line)
            if match:
                data['dob'] = fmt_date(match.group())
        elif "Date" in line and "DOB" not in line:
            match = re.search(r'(\d{2}/\d{2}/\d{4})', line)
            if match:
                data['date'] = fmt_date(match.group())
        elif "INJECTION" in line:
            data['injection'] = "Yes" if "YES" in line.upper() else "No"
        elif "Exercise Therapy" in line:
            data['exercise_therapy'] = "Yes" if "YES" in line.upper() else "No"
        for key in difficulty_keys:
            if key.lower() in line.lower():
                key_formatted = key.lower().replace(' ', '_')
                value = [int(num) for num in re.findall(r'\d+', line)]
                data['difficulty_ratings'][key_formatted] = value[-1] if value else 0
        if "since last treatment" in line.lower():
            data['patient_changes']['since_last_treatment'] = line.split(':')[-1].strip()
        if "since the start of treatment" in line.lower():
            data['patient_changes']['since_start_of_treatment'] = line.split(':')[-1].strip()
        if "last three days" in line.lower():
            data['patient_changes']['last_3_days'] = line.split(':')[-1].strip()
        for key in pain_keys:
            if key.lower() in line.lower():
                key_formatted = key.lower().replace(' ', '_')
                value = [int(num) for num in re.findall(r'\d+', line)]
                data['pain_symptoms'][key_formatted] = value[-1] if value else 0
    return data

def insert_data(data):
    try:
        import mysql.connector
        connection = mysql.connector.connect(
            host='localhost',
            user='root',
            password='root',
            database='ocr'
        )
        cursor = connection.cursor()
        insert_patient = "INSERT INTO patients (name, dob) VALUES (%s, %s)"
        dob_value = data.get('dob') if data.get('dob') else None
        cursor.execute(insert_patient, (data.get('patient_name'), dob_value))
        patient_id = cursor.lastrowid
        insert_form = "INSERT INTO forms_data (patient_id, form_json) VALUES (%s, %s)"
        cursor.execute(insert_form, (patient_id, json.dumps(data)))
        connection.commit()
        print("Data inserted successfully.")
    except Exception as err:
        print(f"Error: {err}")
    finally:
        if connection.is_connected():
            cursor.close()
            connection.close()

if __name__ == "__main__":
    img_path = r"D:\img.png"
    if os.path.exists(img_path):
        txt, nums = get_txt(img_path)
        structured_data = parse(txt, nums)
        with open('output.json', 'w') as json_file:
            json.dump(structured_data, json_file, indent=4)
        insert_data(structured_data)
        print(json.dumps(structured_data, indent=4))
    else:
        print("Image file not found.")


Data inserted successfully.
{
    "patient_name": "Sift",
    "dob": "",
    "date": "",
    "injection": "Yes",
    "exercise_therapy": "",
    "difficulty_ratings": {
        "bending": 2345,
        "putting_on_shoes": 45,
        "sleeping": 12345,
        "standing": 45,
        "walking": 34,
        "driving": 4,
        "preparing": 12345,
        "yard_work": 45,
        "picking_up_items": 1234
    },
    "patient_changes": {
        "since_last_treatment": ""
    },
    "pain_symptoms": {
        "pain": 2,
        "numbness": 2,
        "tingling": 2,
        "burning": 2,
        "tightness": 2
    },
    "medical_assistant_data": {},
    "filled_by_ma": "",
    "medical_assistant_data filled ny Ma": {
        "blood_pressure": "135/89",
        "hr": 90,
        "weight": 90,
        "height": "5'8",
        "spo2": 88,
        "temperature": "97.6",
        "blood_glucose": 163,
        "respirations": 15
    }
}
